In [ ]:
# ==============================================================
# DEPRIMAP — WRI (p_informal) vs RF Deprivation Statistics
# ==============================================================
# Computes: total segments, RF-deprived segments, WRI-deprived segments,
#            total population, RF-deprived population, WRI-deprived population
# across τ = 0.1, 0.2, 0.3 thresholds.
# ==============================================================

from pathlib import Path
from joblib import Parallel, delayed
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio import mask as rio_mask
from shapely.geometry import box

In [ ]:
# --------------------------------------------------------------
# 1️⃣ CONFIGURATION
# --------------------------------------------------------------
RF_DIR = Path(r"D:\VSG\DIRTY_MODEL\February2026\csmd_model_v2\2_modelling\02_application\predictions")
WRI_ROOT = Path(r"D:\VSG\DIRTY_MODEL\February2026\csmd_model_v2\data\raw\WRI\PerCountry_Files")
OUT_CSV = Path(r"D:\VSG\DIRTY_MODEL\February2026\csmd_model_v2\3_comparitive_analysis\WRI\Outputs\wri_rf_population_stats.csv")

TAUS = [0.1, 0.2, 0.3]
MIN_VALID_PX = 1
LULC_INFORMAL_VALUES = {2, 3}  # pixel values representing informal areas
N_JOBS = -1                    # use all cores

In [ ]:
# --------------------------------------------------------------
# 2️⃣ HELPER FUNCTION: compute p_informal for one polygon
# --------------------------------------------------------------
def compute_p_informal(ds, geom):
    """Return fraction of LULC pixels == 2 or 3 within geom."""
    try:
        arr, _ = rio_mask.mask(ds, [geom.__geo_interface__], crop=True, filled=False, indexes=1)
        arr = arr[0] if arr.ndim == 3 else arr
    except ValueError:
        return np.nan, 0
    if hasattr(arr, "mask"):
        valid = ~arr.mask
        vals = arr.data[valid]
    else:
        vals = arr
    if vals.size == 0:
        return np.nan, 0
    valid_px = len(vals)
    if valid_px < MIN_VALID_PX:
        return np.nan, 0
    p_inf = float(np.count_nonzero(np.isin(vals, list(LULC_INFORMAL_VALUES)))) / valid_px
    return p_inf, valid_px

In [ ]:
# --------------------------------------------------------------
# 3️⃣ PER-COUNTRY PROCESSOR
# --------------------------------------------------------------
def process_country(country):
    print(f"▶ {country}")
    gpkg_path = RF_DIR / f"{country}_rf_preds.gpkg"
    wri_dir = WRI_ROOT / country
    if not gpkg_path.exists():
        return {"country": country, "status": "missing_gpkg"}
    if not wri_dir.exists():
        return {"country": country, "status": "missing_wri"}

    # Find a WRI raster (first *.tif)
    tif_files = sorted(wri_dir.glob("*.tif"))
    if not tif_files:
        return {"country": country, "status": "no_tif"}
    tif_path = tif_files[0]

    # Load data
    try:
        gdf = gpd.read_file(gpkg_path)[["geometry", "rf_label", "POP_SEG"]].copy()
    except Exception as e:
        return {"country": country, "status": f"read_error: {e}"}

    with rasterio.open(tif_path) as ds:
        r_crs = ds.crs
        bounds = box(*ds.bounds)

        # Align CRS
        if gdf.crs != r_crs:
            gdf = gdf.to_crs(r_crs)

        # Filter out polygons completely outside raster
        gdf = gdf[gdf.intersects(bounds)].copy()

        # Compute mean p_informal per segment
        p_vals = []
        for geom in gdf.geometry:
            p, vpx = compute_p_informal(ds, geom)
            p_vals.append(p)
        gdf["p_informal"] = p_vals

    # Remove missing
    gdf = gdf.dropna(subset=["rf_label", "p_informal", "POP_SEG"]).copy()
    gdf["rf_label"] = gdf["rf_label"].astype(int)
    gdf["POP_SEG"] = gdf["POP_SEG"].astype(float)

    total_segments = len(gdf)
    total_pop = gdf["POP_SEG"].sum()
    rf_dep_segments = int((gdf["rf_label"] == 1).sum())
    rf_dep_pop = gdf.loc[gdf["rf_label"] == 1, "POP_SEG"].sum()

    rows = []
    for tau in TAUS:
        wri_dep = gdf["p_informal"] >= tau
        wri_dep_segments = int(wri_dep.sum())
        wri_dep_pop = gdf.loc[wri_dep, "POP_SEG"].sum()

        rows.append({
            "country": country,
            "threshold": tau,
            "total_segments": total_segments,
            "rf_deprived_segments": rf_dep_segments,
            "wri_deprived_segments": wri_dep_segments,
            "total_population": total_pop,
            "rf_deprived_population": rf_dep_pop,
            "wri_deprived_population": wri_dep_pop,
        })
    return rows

In [ ]:
# --------------------------------------------------------------
# 4️⃣ RUN ALL COUNTRIES IN PARALLEL
# --------------------------------------------------------------
countries = sorted([p.name for p in WRI_ROOT.iterdir() if p.is_dir()])
print(f"Found {len(countries)} countries: {countries[:6]}{' ...' if len(countries) > 6 else ''}")

results = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(process_country)(c) for c in countries
)

# Flatten and clean
rows = [r for sub in results if isinstance(sub, list) for r in sub]
df = pd.DataFrame(rows)
df = df.dropna(subset=["threshold"])

In [ ]:
# --------------------------------------------------------------
# 5️⃣ ADD GLOBAL SUMMARY ROW
# --------------------------------------------------------------
global_rows = []
for tau in TAUS:
    sub = df[df["threshold"] == tau]
    global_rows.append({
        "country": "GLOBAL",
        "threshold": tau,
        "total_segments": sub["total_segments"].sum(),
        "rf_deprived_segments": sub["rf_deprived_segments"].sum(),
        "wri_deprived_segments": sub["wri_deprived_segments"].sum(),
        "total_population": sub["total_population"].sum(),
        "rf_deprived_population": sub["rf_deprived_population"].sum(),
        "wri_deprived_population": sub["wri_deprived_population"].sum(),
    })
global_df = pd.DataFrame(global_rows)

final = pd.concat([df, global_df], ignore_index=True)

In [ ]:
# --------------------------------------------------------------
# 6️⃣ SAVE OUTPUT
# --------------------------------------------------------------
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(OUT_CSV, index=False)
print(f"\n✅ Saved consolidated statistics to:\n{OUT_CSV}")

# Optional preview
print("\nSample output:")
display(final.head(12))

In [1]:
# ============================================================
# Global Summary Table — WRI vs RF (p_informal)
# ============================================================
import pandas as pd
from pathlib import Path

# --- Paths ---
IN_CSV  = Path(r"D:\VSG\DIRTY_MODEL\February2026\csmd_model_v2\3_comparitive_analysis\WRI\Outputs\wri_rf_population_stats.csv")
OUT_CSV = IN_CSV.parent / "wri_rf_population_summary_GLOBAL_rule_threshold_table_millions.csv"

# --- Load data ---
df = pd.read_csv(IN_CSV)

# --- Keep only needed columns ---
keep_cols = [
    "country",
    "threshold",
    "total_segments",
    "rf_deprived_segments",
    "wri_deprived_segments",
    "total_population",
    "rf_deprived_population",
    "wri_deprived_population"
]
df = df[keep_cols].copy()

# --- Global aggregation (sum across countries) ---
global_summary = (
    df.groupby("threshold", as_index=False)
      .agg({
          "total_segments": "sum",
          "rf_deprived_segments": "sum",
          "wri_deprived_segments": "sum",
          "total_population": "sum",
          "rf_deprived_population": "sum",
          "wri_deprived_population": "sum"
      })
)

# --- Convert populations to millions and calculate shares ---
rows = []
for _, row in global_summary.iterrows():
    tau = row["threshold"]
    rows.append({
        "Rule": "WRI (p_informal)",
        "Threshold": tau,
        "TotalSegments": int(row["total_segments"]),
        "RF_Deprived_Seg": int(row["rf_deprived_segments"]),
        "Rule_Deprived_Seg": int(row["wri_deprived_segments"]),
        "Total_Pop_M": round(row["total_population"] / 1e6, 2),
        "RF_Deprived_Pop_M": round(row["rf_deprived_population"] / 1e6, 2),
        "Rule_Deprived_Pop_M": round(row["wri_deprived_population"] / 1e6, 2),
    })

out = pd.DataFrame(rows).sort_values("Threshold").reset_index(drop=True)

# --- Add % shares ---
out["RF_Deprived_Pop_%"] = (out["RF_Deprived_Pop_M"] / out["Total_Pop_M"] * 100).round(2)
out["Rule_Deprived_Pop_%"] = (out["Rule_Deprived_Pop_M"] / out["Total_Pop_M"] * 100).round(2)

# --- Save and preview ---
out.to_csv(OUT_CSV, index=False)

print("\n✅ Global WRI–RF Summary Table (Millions):\n")
print(out.to_string(index=False))
print(f"\n✅ Saved to: {OUT_CSV}")



✅ Global WRI–RF Summary Table (Millions):

            Rule  Threshold  TotalSegments  RF_Deprived_Seg  Rule_Deprived_Seg  Total_Pop_M  RF_Deprived_Pop_M  Rule_Deprived_Pop_M  RF_Deprived_Pop_%  Rule_Deprived_Pop_%
WRI (p_informal)        0.1         335782           107060             163248       477.58             140.78               220.80              29.48                46.23
WRI (p_informal)        0.2         335782           107060             146888       477.58             140.78               186.23              29.48                38.99
WRI (p_informal)        0.3         335782           107060             134506       477.58             140.78               163.20              29.48                34.17

✅ Saved to: D:\VSG\DIRTY_MODEL\February2026\csmd_model_v2\3_comparitive_analysis\WRI\Outputs\wri_rf_population_summary_GLOBAL_rule_threshold_table_millions.csv
